# Self-Attention · Transformer · Transformer 파생 모델(BERT/GPT/T5) 정리

수업 자료 "SelfAttention과 Transformer.pdf" + "Transformer 파생형 모델.pdf" 필터링 요약 + 실무 코드

**필터링 기준**: RNN/LSTM/Seq2seq는 이미 이전 실습(RNN 챗봇)에서 다뤘고 지금 배우는 커리큘럼의 핵심도 아니므로 아주 짧게만 복습한다.
반대로 **Self-Attention과 Transformer는 지금 커리큘럼의 진짜 핵심이자 실무에서 매일 쓰는 내용**이므로 코드 포함해서 자세히 다룬다.
BERT/GPT/T5도 마찬가지로 실무 비중이 매우 높아서(사실상 요즘 NLP 작업 대부분이 이 셋 중 하나의 변형을 가져다 쓰는 것) 개념 + 실전 코드를 둘 다 다룬다.

In [ ]:
# 필요한 패키지 설치 (최초 1회만 실행)
# !pip install torch transformers nltk


## 1. RNN → LSTM → Seq2seq 초고속 복습 (코드 없이 요약만)

이미 이전 실습(RNN 기반 챗봇)에서 직접 만들어봤으므로 여기서는 "왜 Self-Attention이 등장했는지"의 배경만 짚고 넘어간다.

- **RNN**: 이전 시점의 hidden state를 다음 시점으로 순차적으로 전달. Cell을 반복할수록 초기 입력 정보가 희석됨(장기 의존성 문제) + 순차 처리라 병렬화 불가능(느림).
- **LSTM**: Cell State(장기 기억)를 따로 두고 망각/입력/출력 게이트로 정보를 선택적으로 유지·삭제해서 장기 의존성 문제를 완화. 하지만 여전히 순차 처리라 느린 건 그대로.
- **Seq2seq**: 인코더가 입력 문장 전체를 **고정 크기의 Context Vector 하나**로 요약하고, 디코더가 그 벡터만 보고 문장을 생성. 문제는 문장이 길어지면 이 벡터 하나에 모든 정보를 우겨넣다 보니 앞부분 정보가 소실됨(예: "마트에 가서 오렌지 두 개를 2000원에 사와"에서 "두 개", "2000원" 같은 디테일이 번역에서 누락됨).

> 핵심 결론: **"모든 시점을 하나의 벡터로 요약하지 말고, 출력을 만들 때마다 입력의 어느 부분이 중요한지 매번 다시 계산하자"** → 이게 Attention이고, 이걸 극단까지 밀어붙여서 RNN 자체를 없애버린 게 Self-Attention / Transformer.

## 2. Self-Attention (Scaled Dot-Product Attention)

**핵심 아이디어**: 문장 속 모든 토큰이 서로에게 "내가 너를 얼마나 참고해야 하는지" 점수(가중치)를 매기고, 그 가중치로 각 토큰의 표현을 다시 계산한다.

### Query, Key, Value
- 기존 Seq2seq의 Attention은 Query=디코더 hidden state, Key/Value=인코더 hidden state로 **서로 다른 문장**에서 왔다.
- **Self-Attention은 Q, K, V가 전부 같은 문장(같은 입력)에서 나온다.** 즉 문장이 "스스로에게" 주목한다.
- 하나의 입력 임베딩 벡터에 3개의 서로 다른 Linear layer($W^Q, W^K, W^V$)를 곱해서 Q, K, V를 각각 만든다.

### 계산 공식 (Scaled Dot-Product Attention)

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- $QK^T$: Query와 Key를 내적(dot product)해서 "얼마나 관련 있는지" 점수(Attention score)를 구함
- $\sqrt{d_k}$로 나누는 이유(Scaling): Q, K의 차원 수가 커질수록 내적 값 자체가 커져서 softmax가 한쪽으로 쏠려버림(gradient가 죽음) → 차원 수의 제곱근으로 나눠서 값의 크기를 안정화
- softmax: 점수를 [0, 1] 확률 분포로 정규화
- 마지막에 V와 곱해서, "중요하다고 판단된 토큰의 Value를 더 많이 반영한" 새로운 표현을 만듦

숫자 예시(PDF 참고): 문장 `['the','train','left','the','station','on','time']`에서 "station"이라는 단어는 "train"(0.8), "left"(0.6)와 관련성이 높게 계산되고, 이 점수들로 가중합을 해서 "station"의 새로운 문맥 인지 벡터(context-aware vector)를 만든다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
  """
  Self-Attention의 핵심 연산: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
  """

  def __init__(self):
    super(ScaledDotProductAttention, self).__init__()
    # 이 모듈 자체는 학습 파라미터가 없음 (Q, K, V를 만드는 Linear layer는 밖에서 이미 적용되어 들어온다고 가정)

  def forward(self, Q, K, V, mask=None):
    """
    Args:
      Q: Query 텐서 [batch_size, n_heads, seq_length, d_k]
      K: Key 텐서   [batch_size, n_heads, seq_length, d_k]
      V: Value 텐서 [batch_size, n_heads, seq_length, d_v]
      mask: 특정 위치를 무시하기 위한 마스크 (패딩 마스크 또는 미래 시점 차단용 마스크)
    Returns:
      output: 가중합된 결과 [batch_size, n_heads, seq_length, d_v]
      attn_weights: 시각화/분석용 Attention 가중치 [batch_size, n_heads, seq_length, seq_length]
    """
    d_k = Q.size(-1)  # Key/Query의 차원 수 (스케일링에 사용)

    # 1) Q와 K의 내적으로 Attention score 계산: [.., seq_len, seq_len] 형태가 됨
    #    K.transpose(-2, -1): 마지막 두 차원을 바꿔서 (d_k, seq_len) 형태로 만들어 행렬곱이 가능하게 함
    scores = torch.matmul(Q, K.transpose(-2, -1))

    # 2) Scaling: 차원 수가 커질수록 내적 값이 커지는 걸 방지 (softmax 포화 방지)
    scores = scores / math.sqrt(d_k)

    # 3) 마스킹 (선택): 마스크가 있는 위치를 -inf에 가깝게 만들어서 softmax 후 거의 0이 되게 함
    #    - 패딩 마스크: 의미 없는 [PAD] 토큰을 무시
    #    - 미래 마스크(subsequent mask): 디코더가 아직 생성 안 된 미래 토큰을 못 보게 차단
    if mask is not None:
      scores = scores.masked_fill(mask == 0, float('-inf'))

    # 4) softmax로 점수를 [0, 1] 확률 분포로 정규화 (마지막 차원 기준)
    attn_weights = F.softmax(scores, dim=-1)

    # 5) 가중치를 Value에 곱해서 최종 context-aware 벡터 생성
    output = torch.matmul(attn_weights, V)

    return output, attn_weights


# ---- 숫자로 확인해보기 ----
torch.manual_seed(0)
batch_size, n_heads, seq_len, d_k = 1, 1, 4, 8

# 임의의 Q, K, V 생성 (실제로는 임베딩에 Linear layer를 곱해서 만들어짐)
Q = torch.randn(batch_size, n_heads, seq_len, d_k)
K = torch.randn(batch_size, n_heads, seq_len, d_k)
V = torch.randn(batch_size, n_heads, seq_len, d_k)

attention = ScaledDotProductAttention()
output, attn_weights = attention(Q, K, V)

print("Attention 가중치 shape:", attn_weights.shape)  # [1, 1, 4, 4] -> 토큰 4개가 서로에게 매긴 점수
print("Attention 가중치 (각 행의 합은 1):\n", attn_weights[0, 0])
print("각 행 합계 (softmax 정규화 확인):", attn_weights[0, 0].sum(dim=-1))
print("출력 벡터 shape:", output.shape)  # [1, 1, 4, 8] -> 입력과 동일한 shape의 문맥 반영 벡터


## 3. Multi-Head Attention

**왜 필요한가**: Self-Attention을 딱 1번만 계산하면 "한 가지 관점"에서만 토큰 간 관계를 보게 된다. 문법적 관계, 의미적 관계, 위치적 관계 등 **여러 관점을 동시에** 보고 싶다.

**아이디어**: 임베딩 차원(d_model)을 head 개수(n_heads)만큼 쪼개서, 각 head가 서로 다른 부분 공간(subspace)에서 독립적으로 Attention을 계산한 뒤, 결과를 다시 이어붙이고(concat) Linear layer로 합친다.

- 여러 모델을 동시에 학습시켜서 앙상블하는 것과 비슷한 효과(다양한 특징을 각 head가 나눠서 학습)
- Head별 연산은 서로 독립적이라 병렬 처리 가능 (RNN과 달리 GPU를 꽉 채워서 쓸 수 있음 = 실무에서 학습 속도가 빠른 핵심 이유)

In [ ]:
class MultiHeadAttention(nn.Module):
  """
  d_model 차원을 n_heads개로 쪼개서 병렬로 Self-Attention을 수행한 뒤 다시 합치는 모듈
  """

  def __init__(self, d_model, n_heads):
    super(MultiHeadAttention, self).__init__()
    assert d_model % n_heads == 0, "d_model은 n_heads로 나누어 떨어져야 함"

    self.d_model = d_model
    self.n_heads = n_heads
    self.d_k = d_model // n_heads  # head 하나가 담당하는 차원 수

    # Q, K, V를 만드는 Linear layer. 입력/출력 모두 d_model 차원 (head로 쪼개는 건 forward에서 reshape로 처리)
    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)

    # 모든 head의 결과를 concat한 뒤 다시 섞어주는 최종 출력 Linear layer
    self.W_o = nn.Linear(d_model, d_model)

    self.attention = ScaledDotProductAttention()

  def split_heads(self, x, batch_size):
    # x: [batch_size, seq_length, d_model] -> [batch_size, n_heads, seq_length, d_k]로 reshape
    x = x.view(batch_size, -1, self.n_heads, self.d_k)
    return x.transpose(1, 2)  # head 차원을 앞으로 보내서 head별 병렬 연산이 되게 함

  def forward(self, query, key, value, mask=None):
    # mask는 [batch_size, 1, seq_q, seq_k] 또는 [1, 1, seq_q, seq_k]처럼
    # head 차원(dim=1)이 1이어야 함 -> 아래 Attention 연산에서 n_heads 차원으로 자동 broadcast됨
    # (create_padding_mask, generate_square_subsequent_mask 모두 이 규칙에 맞춰 shape을 맞춰서 넘겨줘야 함)
    batch_size = query.size(0)

    # 1) Q, K, V 생성 후 head 수만큼 쪼갬
    Q = self.split_heads(self.W_q(query), batch_size)
    K = self.split_heads(self.W_k(key), batch_size)
    V = self.split_heads(self.W_v(value), batch_size)

    # 2) 각 head별로 독립적으로 Scaled Dot-Product Attention 수행
    attn_output, attn_weights = self.attention(Q, K, V, mask)

    # 4) 다시 원래 shape로 합치기: [batch, n_heads, seq, d_k] -> [batch, seq, d_model]
    attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

    # 5) 최종 Linear layer로 head별 정보를 한번 더 섞어줌
    output = self.W_o(attn_output)

    return output, attn_weights


# ---- 확인 ----
d_model, n_heads = 512, 8
mha = MultiHeadAttention(d_model, n_heads)

x = torch.randn(2, 10, d_model)  # [batch_size=2, seq_length=10, d_model=512]
out, weights = mha(x, x, x)      # Self-Attention이므로 Q, K, V 자리에 전부 같은 x를 넣음

print("MultiHeadAttention 출력 shape:", out.shape)          # [2, 10, 512] -> 입력과 동일
print("head별 Attention 가중치 shape:", weights.shape)      # [2, 8, 10, 10] -> head 8개


## 4. Positional Encoding

**문제**: Transformer는 RNN과 달리 토큰을 순서대로 하나씩 넣지 않고 **문장 전체를 한 번에** 입력한다. 그래서 "몇 번째 단어인지"에 대한 순서 정보가 구조적으로 존재하지 않는다.

**해결**: 단어 임베딩 벡터에, 위치마다 고유한 값을 갖는 "위치 인코딩 벡터"를 **더해서(sum)** 순서 정보를 주입한다. sin/cos 함수를 이용해 고정된(학습되지 않는) 벡터를 만든다.

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

- $pos$: 문장 내 토큰 위치(0, 1, 2, ...)
- $i$: 임베딩 벡터 내 차원 인덱스
- 짝수 차원엔 sin, 홀수 차원엔 cos를 사용 → 위치마다 고유한 패턴이 생기고, 상대적 위치 관계도 학습 가능

In [ ]:
class PositionalEncoding(nn.Module):
  """
  sin/cos 기반 고정 위치 인코딩. 학습되는 파라미터가 없다.
  """

  def __init__(self, d_model, max_seq_length=5000):
    super(PositionalEncoding, self).__init__()

    # [max_seq_length, d_model] 크기의 빈 위치 인코딩 테이블을 미리 만들어둠
    pe = torch.zeros(max_seq_length, d_model)

    # 0 ~ max_seq_length-1 까지의 위치 번호를 세로 벡터로: [max_seq_length, 1]
    position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)

    # 10000^(2i/d_model) 의 역수를 지수 함수 형태로 안전하게 계산 (log-exp 트릭으로 오버플로 방지)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

    # 짝수 인덱스 차원엔 sin, 홀수 인덱스 차원엔 cos 적용
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    # [1, max_seq_length, d_model] 형태로 batch 차원을 추가해서 broadcasting이 되게 함
    pe = pe.unsqueeze(0)

    # 학습되지 않는 상수이므로 파라미터가 아니라 buffer로 등록 (모델 저장/로드는 되지만 gradient는 안 흐름)
    self.register_buffer('pe', pe)

  def forward(self, x):
    # x: [batch_size, seq_length, d_model]
    # 입력 문장 길이만큼만 잘라서 임베딩에 더해줌
    return x + self.pe[:, :x.size(1)]


# ---- 확인 ----
pos_enc = PositionalEncoding(d_model=512)
word_embedding = torch.randn(2, 10, 512)  # [batch_size=2, seq_length=10, d_model=512]
embedded_with_position = pos_enc(word_embedding)

print("위치 인코딩 적용 전후 shape 동일 여부:", word_embedding.shape == embedded_with_position.shape)
print("0번째 토큰 위치 인코딩 앞부분 5개 값:", pos_enc.pe[0, 0, :5])
print("1번째 토큰 위치 인코딩 앞부분 5개 값:", pos_enc.pe[0, 1, :5])


## 5. Feed Forward Network + Encoder 조립

Attention이 "토큰 간 관계"를 계산하는 역할이라면, **Position-wise Feed Forward Network(FFN)**는 각 토큰 위치별로 독립적으로 비선형 변환을 가해서 표현력을 높이는 역할이다(Linear → ReLU → Linear).

Encoder 한 층(EncoderLayer)은 다음 순서로 구성된다: **Multi-Head Self-Attention → Add & Norm(잔차 연결 + 정규화) → FFN → Add & Norm**. 이 층을 N번(논문 기준 6번) 쌓은 것이 전체 Encoder.

- **Add(잔차 연결, Residual Connection)**: 입력을 출력에 그대로 더해줘서, 층이 깊어져도 gradient가 잘 흐르게 함(LSTM의 기울기 소실 문제와 비슷한 고민을 여기서도 함)
- **Norm(Layer Normalization)**: 값의 분포를 안정시켜서 학습을 안정적으로 만듦

In [ ]:
class PositionwiseFeedForward(nn.Module):
  """
  토큰 위치마다 독립적으로 적용되는 2단 Linear 변환: Linear -> ReLU -> Linear
  """

  def __init__(self, d_model, d_ff, dropout=0.1):
    super(PositionwiseFeedForward, self).__init__()
    self.linear1 = nn.Linear(d_model, d_ff)   # 보통 d_ff는 d_model의 4배 정도로 크게 키움 (표현력 확보)
    self.linear2 = nn.Linear(d_ff, d_model)   # 다시 원래 차원으로 축소
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    x = F.relu(self.linear1(x))  # 비선형성 추가
    x = self.dropout(x)
    return self.linear2(x)


class EncoderLayer(nn.Module):
  """
  Multi-Head Self-Attention + FFN을 Add&Norm(잔차 연결 + LayerNorm)으로 감싼 Encoder 한 층
  """

  def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
    super(EncoderLayer, self).__init__()
    self.self_attn = MultiHeadAttention(d_model, n_heads)
    self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, mask=None):
    # 1) Self-Attention: Q, K, V 모두 x (자기 자신)
    attn_output, _ = self.self_attn(x, x, x, mask)
    # 2) 잔차 연결(입력 x를 그대로 더함) + LayerNorm
    x = self.norm1(x + self.dropout(attn_output))

    # 3) Feed Forward
    ff_output = self.feed_forward(x)
    # 4) 다시 잔차 연결 + LayerNorm
    x = self.norm2(x + self.dropout(ff_output))

    return x


class Encoder(nn.Module):
  """
  임베딩 + 위치 인코딩 + EncoderLayer를 N개 쌓은 전체 Encoder
  """

  def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1, max_seq_length=5000):
    super(Encoder, self).__init__()
    self.embedding = nn.Embedding(vocab_size, d_model)
    self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
    self.dropout = nn.Dropout(dropout)

    # EncoderLayer를 n_layers개 만큼 쌓음 (논문 기본값 6개)
    self.layers = nn.ModuleList([
      EncoderLayer(d_model, n_heads, d_ff, dropout)
      for _ in range(n_layers)
    ])

  def forward(self, x, mask=None):
    # 임베딩 값의 크기를 sqrt(d_model)만큼 키워줌 (논문에서 사용한 스케일링, 위치 인코딩과 크기 균형을 맞추기 위함)
    x = self.embedding(x) * math.sqrt(self.embedding.embedding_dim)
    x = self.positional_encoding(x)
    x = self.dropout(x)

    for layer in self.layers:
      x = layer(x, mask)

    return x


# ---- 확인 ----
encoder = Encoder(vocab_size=1000, d_model=512, n_layers=2, n_heads=8, d_ff=2048)
dummy_input = torch.randint(0, 1000, (2, 10))  # [batch_size=2, seq_length=10] 형태의 토큰 id 시퀀스
encoder_output = encoder(dummy_input)
print("Encoder 출력 shape:", encoder_output.shape)  # [2, 10, 512]


## 6. 마스킹(Masking)

Transformer 디코더는 학습할 때 정답 문장 전체를 한 번에 넣는데, 그렇다고 미래 시점의 단어를 미리 커닝하면 안 된다. 그래서 **두 가지 마스크**가 필요하다.

- **패딩 마스크(Padding mask)**: 배치 안에서 문장 길이를 맞추려고 채운 `[PAD]` 토큰은 Attention 계산에서 무시
- **미래 마스크(Subsequent mask / Look-ahead mask)**: 디코더가 $t$번째 단어를 예측할 때 $t+1$번째 이후 단어는 못 보게 차단 (하삼각행렬 형태)

In [ ]:
def create_padding_mask(seq, pad_idx=0):
  """
  Args:
    seq: 토큰 id 시퀀스 [batch_size, seq_length]
    pad_idx: 패딩 토큰의 인덱스
  Returns:
    패딩 마스크 [batch_size, 1, 1, seq_length] (1=유효 토큰, 0=패딩 토큰)
  """
  # 패딩 토큰이 아닌 위치는 True(1), 패딩인 위치는 False(0)
  mask = (seq != pad_idx).unsqueeze(1).unsqueeze(2)
  return mask


def generate_square_subsequent_mask(size):
  """
  Args:
    size: 시퀀스 길이
  Returns:
    하삼각행렬 마스크 [size, size]. 대각선 포함 아래쪽만 1(허용), 위쪽(미래)은 0(차단)
  """
  # torch.triu: 상삼각행렬(대각선 위쪽)을 1로 채움 -> 이걸 뒤집어서 "허용/차단" 논리 마스크로 사용
  mask = torch.tril(torch.ones(size, size)).bool()
  return mask


# ---- 확인 ----
seq = torch.tensor([[5, 7, 2, 0, 0], [3, 8, 1, 4, 0]])  # 0이 패딩 토큰이라고 가정
pad_mask = create_padding_mask(seq, pad_idx=0)
print("패딩 마스크 shape:", pad_mask.shape)
print("첫 번째 문장 패딩 마스크:", pad_mask[0, 0, 0])  # [True, True, True, False, False]

sub_mask = generate_square_subsequent_mask(5)
print("미래 마스크 (하삼각행렬):\n", sub_mask)
# 1번째 행: 자기 자신만 볼 수 있음 / 마지막 행: 전체를 다 볼 수 있음(이미 다 생성된 상태이므로)


## 7. Decoder와 전체 Transformer 조립

Decoder 한 층은 Encoder보다 블록이 하나 더 많다: **① Masked Multi-Head Self-Attention(미래 마스크 사용, 타깃 문장 내부 문맥 학습) → ② Multi-Head Cross-Attention(Query=디코더, Key/Value=인코더 출력 → 타깃이 소스 문장 어디를 참고할지 학습) → ③ FFN**, 각 단계마다 Add & Norm.

이 구조가 실제 수업 PDF에 있는 instructor 코드와 동일하다(아래는 그 코드에 라인별 주석만 추가한 버전).

In [ ]:
class DecoderLayer(nn.Module):
  """
  Masked Self-Attention + Cross-Attention + FFN으로 구성된 Decoder 한 층
  """

  def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
    super(DecoderLayer, self).__init__()
    self.self_attn = MultiHeadAttention(d_model, n_heads)    # ① 타깃 문장 내부를 보는 Masked Self-Attention
    self.cross_attn = MultiHeadAttention(d_model, n_heads)   # ② 인코더 출력을 참고하는 Cross-Attention
    self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.norm3 = nn.LayerNorm(d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, enc_output, tgt_mask=None, src_tgt_mask=None):
    # 1) Masked Self-Attention: Q=K=V=x(타깃 문장), 미래 시점은 tgt_mask로 차단
    attn_output, _ = self.self_attn(x, x, x, tgt_mask)
    x = self.norm1(x + self.dropout(attn_output))

    # 2) Cross-Attention: Query는 디코더(x), Key/Value는 인코더 출력(enc_output)
    #    -> "지금 만들고 있는 타깃 단어가 소스 문장의 어느 부분을 참고해야 하는지" 학습
    attn_output, _ = self.cross_attn(x, enc_output, enc_output, src_tgt_mask)
    x = self.norm2(x + self.dropout(attn_output))

    # 3) Feed Forward
    ff_output = self.feed_forward(x)
    x = self.norm3(x + self.dropout(ff_output))

    return x


class Decoder(nn.Module):
  """
  트랜스포머 디코더
  임베딩, 위치 인코딩, 다수의 디코더 레이어로 구성
  """

  def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1, max_seq_length=5000):
    super(Decoder, self).__init__()
    self.embedding = nn.Embedding(vocab_size, d_model)
    self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
    self.dropout = nn.Dropout(p=dropout)

    # 여러 개의 디코더 레이어 스택
    self.layers = nn.ModuleList([
      DecoderLayer(d_model, n_heads, d_ff, dropout)
      for _ in range(n_layers)
    ])

  def forward(self, x, enc_output, tgt_mask=None, src_tgt_mask=None):
    """
    Args:
      x: 입력 시퀀스 [batch_size, tgt_seq_length]
      enc_output: 인코더 출력 [batch_size, src_seq_length, d_model]
      tgt_mask: 타겟 시퀀스 마스크 [batch_size, 1, tgt_seq_length, tgt_seq_length]
      src_tgt_mask: 소스-타겟 어텐션 마스크 [batch_size, 1, tgt_seq_length, src_seq_length]
    Returns:
      디코더 출력 [batch_size, tgt_seq_length, d_model]
    """
    # 임베딩 및 위치 인코딩 적용
    x = self.embedding(x) * math.sqrt(self.embedding.embedding_dim)
    x = self.positional_encoding(x)
    x = self.dropout(x)

    # 모든 디코더 레이어를 통과
    for layer in self.layers:
      x = layer(x, enc_output, tgt_mask, src_tgt_mask)

    return x


class Transformer(nn.Module):
  """
  트랜스포머 모델
  인코더, 디코더, 출력 레이어로 구성된 전체 모델
  """

  def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, n_layers=6,
               n_heads=8, d_ff=2048, dropout=0.1, max_seq_length=5000):
    super(Transformer, self).__init__()

    # 모델 구성 요소 및 하이퍼파라미터
    self.d_model = d_model
    self.encoder = Encoder(src_vocab_size, d_model, n_layers, n_heads, d_ff, dropout, max_seq_length)
    self.decoder = Decoder(tgt_vocab_size, d_model, n_layers, n_heads, d_ff, dropout, max_seq_length)

    # 출력 레이어 (선형 변환 + 소프트맥스)
    self.output_layer = nn.Linear(d_model, tgt_vocab_size)

  def forward(self, src, tgt, src_mask=None, tgt_mask=None, src_tgt_mask=None):
    """
    Args:
      src: 소스 시퀀스 [batch_size, src_seq_length]
      tgt: 타겟 시퀀스 [batch_size, tgt_seq_length]
      src_mask: 소스 시퀀스 패딩 마스크
      tgt_mask: 타겟 시퀀스 마스크 (미래 정보 차단)
      src_tgt_mask: 인코더-디코더 어텐션 마스크
    Returns:
      출력 확률 분포 [batch_size, tgt_seq_length, tgt_vocab_size]
    """
    # 인코더 통과
    enc_output = self.encoder(src, src_mask)

    # 디코더 통과
    dec_output = self.decoder(tgt, enc_output, tgt_mask, src_tgt_mask)

    # 출력 레이어 통과 (여기서는 raw logit만 반환. 학습 시 nn.CrossEntropyLoss가 내부적으로 softmax 처리)
    output = self.output_layer(dec_output)

    return output


# ---- 작은 번역 모델처럼 통짜로 돌려보기 ----
src_vocab_size, tgt_vocab_size = 1000, 1200
model = Transformer(src_vocab_size, tgt_vocab_size, d_model=128, n_layers=2, n_heads=4, d_ff=256)

src = torch.randint(1, src_vocab_size, (2, 7))   # [batch_size=2, src_len=7]
tgt = torch.randint(1, tgt_vocab_size, (2, 5))   # [batch_size=2, tgt_len=5]

# generate_square_subsequent_mask는 [tgt_len, tgt_len] 2차원만 반환하므로,
# MultiHeadAttention 내부에서 (batch, head) 차원에 브로드캐스팅되도록 앞에 차원을 2개 추가해줘야 함
# [tgt_len, tgt_len] -> [1, 1, tgt_len, tgt_len]
tgt_mask = generate_square_subsequent_mask(tgt.size(1)).unsqueeze(0).unsqueeze(0)

logits = model(src, tgt, tgt_mask=tgt_mask)
print("Transformer 최종 출력 shape:", logits.shape)  # [2, 5, 1200] -> 타깃 위치마다 단어 사전 크기만큼의 점수
print("각 위치에서 가장 확률 높은 단어 id:", logits.argmax(dim=-1))


## 8. 실무 관점: 직접 구현 vs 사전학습 모델(HuggingFace) 사용

위에서 만든 것처럼 Transformer를 처음부터 직접 구현하는 일은 **실무에서는 거의 없다.** 이미 대규모 데이터로 사전학습된 BERT/GPT/T5 같은 모델을 `transformers` 라이브러리로 가져다 쓰는 게 표준이다.

그래도 내부 구조(Q,K,V, 마스킹, Encoder/Decoder 조립)를 알아야 하는 이유:
- 왜 특정 태스크에 Encoder-only(BERT) / Decoder-only(GPT) / Encoder-Decoder(T5)를 골라야 하는지 판단 가능
- 파인튜닝 중 발생하는 shape 에러, 마스크 관련 버그를 디버깅할 수 있음
- 면접에서 자주 물어봄

아래부터는 두 번째 자료("Transformer 파생형 모델")에서 다룬 BERT/GPT/T5를 실제로 어떻게 코드로 쓰는지 정리한다.

## 9. BERT (Encoder-Only)

**Bidirectional Encoder Representations from Transformer** — Transformer의 **인코더만** 떼어내서 쌓은 모델.

- **양방향(Bidirectional)**: RNN 계열이나 GPT(Decoder)는 앞→뒤 방향으로만 문맥을 보는데, BERT는 Self-Attention 구조 덕분에 문장 전체(양쪽 방향)를 한 번에 보고 각 토큰을 인코딩함.
- **사전학습(Pretraining) 방식 2가지**
  - **MLM(Masked Language Model)**: 문장에서 토큰의 15%를 무작위로 `[MASK]`로 가리고, 그 토큰이 원래 뭐였는지 맞히게 학습 → 문맥 기반 단어 의미를 깊게 학습
  - **NSP(Next Sentence Prediction)**: 두 문장을 이어붙여 입력하고, 두 번째 문장이 실제로 첫 번째 문장 다음에 오는 문장인지(IsNext) 아닌지(NotNext)를 이진 분류로 맞히게 학습
- **활용처**: 텍스트 생성이 아니라 **분류(감정분석, 문서분류), 회귀(문장 유사도), 문장 임베딩 추출**에 주로 사용 (텍스트 생성은 Decoder 계열이 담당)
- **실무 팁**: RoBERTa(NSP 제거 + 더 큰 데이터), DistilBERT(경량화), ALBERT(파라미터 공유로 경량화), KoBERT(한국어 버전) 등 파생 모델이 많음. 상황에 맞게 골라 쓰면 됨.

In [ ]:
# 실무 코드: HuggingFace transformers로 BERT의 MLM(빈칸 채우기) 기능 직접 체험
from transformers import pipeline

# fill-mask 파이프라인: BERT가 사전학습 때 풀었던 것과 똑같은 태스크(마스킹된 토큰 맞히기)
fill_mask = pipeline("fill-mask", model="bert-base-uncased")

result = fill_mask("The capital of France is [MASK].")
for r in result[:3]:
  # r['sequence']: 채워진 전체 문장, r['score']: 모델이 매긴 확률
  print(f"{r['score']:.4f}  {r['sequence']}")


In [ ]:
# 실무 코드: BERT를 문장 분류(감정 분석)에 파인튜닝된 형태로 바로 사용
from transformers import pipeline

# 이미 감정분석 태스크로 파인튜닝된 BERT 계열 모델을 그대로 로드
classifier = pipeline("sentiment-analysis")  # 기본값: distilbert-base-uncased-finetuned-sst-2-english

texts = [
  "This movie was absolutely wonderful, I loved every minute of it!",
  "The service was terrible and the food was cold.",
]

for text in texts:
  result = classifier(text)[0]
  # result = {'label': 'POSITIVE'/'NEGATIVE', 'score': 확신도}
  print(f"[{result['label']} ({result['score']:.4f})] {text}")


In [ ]:
# 실무 코드: BERT로 문장 임베딩을 뽑아서 문장 유사도 계산 (Sentence-level embedding)
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()  # 추론 모드 (dropout 비활성화)

def get_sentence_embedding(sentence):
  # 문장을 토큰화 + BERT 입력 형식으로 변환 (padding/truncation 포함)
  inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)

  with torch.no_grad():  # 추론이므로 gradient 계산 불필요 (메모리/속도 절약)
    outputs = model(**inputs)

  # outputs.last_hidden_state: [1, seq_len, hidden_dim] -> 토큰별 벡터
  # Mean Pooling: 문장 전체를 대표하는 벡터 하나로 평균냄 (CLS 토큰만 쓰는 방법도 있음)
  embedding = outputs.last_hidden_state.mean(dim=1)  # [1, hidden_dim]
  return embedding

sent1 = get_sentence_embedding("What is gravity?")
sent2 = get_sentence_embedding("Why is the sky blue?")
sent3 = get_sentence_embedding("What is gravity in physics?")

# 코사인 유사도로 문장끼리 얼마나 비슷한 의미인지 비교
sim_1_2 = F.cosine_similarity(sent1, sent2).item()
sim_1_3 = F.cosine_similarity(sent1, sent3).item()

print(f"'gravity' vs '하늘이 파란 이유' 유사도: {sim_1_2:.4f}")
print(f"'gravity' vs 'gravity in physics' 유사도: {sim_1_3:.4f}  (당연히 이게 더 높아야 함)")


## 10. GPT / Decoder-Only Models

Transformer의 **디코더 블록만** 연속으로 쌓은 구조. Masked Multi-Head Attention만 사용하고(Cross-Attention 블록이 없음 — 참고할 별도의 인코더가 없으므로), 목적은 오직 **다음 토큰 예측을 통한 텍스트 생성**이다.

### Autoregressive(자기회귀) 생성 원리
- 언어모델은 다음 토큰 생성을 $P(\text{현재 토큰} \mid \text{지금까지 입력된 시퀀스})$ 확률 문제로 접근한다.
- 동작 순서: ① 토큰화된 Prompt(PREFILL)를 모델에 입력 → ② 다음 토큰 1개 생성 → ③ (Prompt + 방금 생성한 토큰)을 다시 입력해서 그 다음 토큰 생성 → ④ 이 과정을 EOS(문장 종결) 토큰이 나올 때까지 반복
- GPT, LLaMA, Claude 등 현존하는 대부분의 LLM이 이 Decoder-only 구조를 따름 (실무적으로 가장 중요한 구조)

In [ ]:
# 실무 코드: GPT-2로 오토리그레시브 텍스트 생성 체험 (라이브러리의 generate() 사용)
from transformers import pipeline, set_seed

set_seed(42)  # 재현 가능한 결과를 위한 시드 고정

generator = pipeline("text-generation", model="gpt2")

prompt = "Artificial intelligence will"
outputs = generator(
  prompt,
  max_length=30,        # 생성할 최대 토큰 길이 (prompt 포함)
  num_return_sequences=2,  # 같은 prompt로 서로 다른 2개 문장 생성
  do_sample=True,       # True면 확률적으로 샘플링(다양성 UP), False면 항상 최고 확률 토큰만 선택(=greedy)
  temperature=0.8,      # 값이 낮을수록 확신에 찬(보수적인) 답, 높을수록 다양하고 창의적인 답
)

for i, output in enumerate(outputs):
  print(f"[생성 {i+1}] {output['generated_text']}")


In [ ]:
# 실무보다는 학습용: Autoregressive 생성 과정을 직접 한 스텝씩 눈으로 확인해보기
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

# 1) 프롬프트를 토큰 id로 변환 (PREFILL 단계)
input_ids = tokenizer.encode("The weather today is", return_tensors="pt")
print("초기 입력 토큰:", tokenizer.convert_ids_to_tokens(input_ids[0]))

# 2) 다음 토큰을 한 개씩 직접 생성해보는 반복문 (내부적으로 model.generate()가 하는 일을 풀어서 쓴 것)
for step in range(5):
  with torch.no_grad():
    outputs = model(input_ids)               # 지금까지의 시퀀스를 통째로 다시 넣음 (자기회귀)
    next_token_logits = outputs.logits[0, -1, :]  # 마지막 위치의 다음 토큰 확률 분포(logit)만 사용

  # argmax로 가장 확률 높은 다음 토큰 id 선택 (여기서는 이해를 위해 greedy 방식 사용)
  next_token_id = torch.argmax(next_token_logits).unsqueeze(0).unsqueeze(0)

  # 3) 기존 시퀀스에 방금 생성한 토큰을 이어붙임 -> 다음 스텝 입력이 됨
  input_ids = torch.cat([input_ids, next_token_id], dim=1)

  print(f"step {step+1}: 다음 토큰 = '{tokenizer.decode(next_token_id[0])}'  "
        f"-> 지금까지: {tokenizer.decode(input_ids[0])}")


## 11. T5 (Text-to-Text Transfer Transformer, Encoder-Decoder)

원본 Transformer와 동일한 **Encoder-Decoder 구조**를 그대로 사용하되, 핵심 아이디어는 **"모든 NLP 문제를 텍스트 → 텍스트 문제로 통일하자"**는 것.

- 번역, 요약, 문장 유사도, 문법성 판단(분류) 등 문제 종류가 달라도 입력에 **Prefix(태스크 지시문)**를 붙여서 전부 "문자열을 넣으면 문자열이 나온다"로 통일함.
  - `"translate English to German: That is good."` → `"Das ist gut."`
  - `"summarize: state authorities dispatched..."` → `"six people hospitalized..."`
- 사전학습: C4(Colossal Clean Crawled Corpus, 750GB)에서 **Span corruption**(문장의 연속된 구간을 통째로 마스킹하고 복원) 방식으로 학습
- Encoder-Decoder 구조는 최근 트렌드에서는 상대적으로 덜 선호됨(구조가 복잡하고, 텍스트 생성만 필요하면 Decoder-only로 충분하기 때문) — 다만 번역처럼 입력/출력이 뚜렷이 구분되는 태스크나, 최근에는 Vision encoder + Text decoder 같은 멀티모달 조합에 자주 활용됨.

In [ ]:
# 실무 코드: T5로 요약 + 번역을 "Prefix만 바꿔서" 같은 모델로 처리하기
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")
model.eval()

def run_t5(task_prefix_and_text, max_new_tokens=40):
  # T5는 입력 앞에 태스크를 알려주는 prefix 문자열을 붙이는 것만으로 태스크가 바뀜
  input_ids = tokenizer(task_prefix_and_text, return_tensors="pt").input_ids

  with torch.no_grad():
    output_ids = model.generate(input_ids, max_new_tokens=max_new_tokens)

  return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 같은 모델, 같은 코드인데 prefix만 바꿔서 완전히 다른 태스크를 수행
translation = run_t5("translate English to German: The weather is nice today.")
summary = run_t5(
  "summarize: state authorities dispatched emergency crews tuesday to survey the damage "
  "after an onslaught of severe weather in mississippi that injured at least 17 people."
)

print("번역 결과:", translation)
print("요약 결과:", summary)


## 12. (참고) LLM 학습 3단계 — 개념만 짧게

실무에서 이 파이프라인을 처음부터 직접 구현할 일은 거의 없지만(대부분 이미 학습된 모델을 가져다 쓰거나 API로 접근), 왜 ChatGPT 같은 모델이 "대화형"으로 대답하는지 이해하려면 알아야 하는 3단계다.

1. **Pretraining(사전학습)**: 인터넷 대량 텍스트로 MLM/NTP(다음 토큰 예측) 방식의 비지도(자기지도) 학습. 단어 간 관계를 확률적으로 학습하지만, 아직은 "빈칸 채우기" 수준만 가능하고 사람 말투로 대답은 못함.
2. **SFT(Supervised Fine-Tuning)**: 사람이 직접 만든 (질문-좋은답변) 쌍 데이터로 지도학습. "질문을 받으면 이렇게 대답해야 한다"는 형식을 가르침.
3. **Alignment(RLHF 등)**: 사람의 선호도(어떤 답변이 더 나은지)를 보상 함수로 만들어서, 모델이 더 안전하고 유용한 답변을 하도록 강화학습으로 미세조정. 최근에는 단계별 추론(Multistep CoT)에 중간 보상을 줘서 추론 능력 자체를 강화하는 방식(OpenAI o1 등)도 활용됨.

> 코드 실습보다는 개념 이해가 중요한 파트라 여기서는 코드를 생략한다.

## 13. LLM/NLP 평가 지표

평가 방식은 크게 3가지로 나뉜다.

1. **통계 기반 평가**: 모델 출력의 통계적 특징만으로 자동 계산 — Perplexity, BLEU, ROUGE, METEOR
2. **벤치마크 평가**: 표준 데이터셋 + 정답(Ground-truth) 기준 — SQuAD(질의응답), MMLU(다분야 지식), GLUE/SuperGLUE(문장 이해), HumanEval(코드 생성)
3. **직접 평가**: 사람 또는 다른 LLM이 답변 품질을 직접 채점 — Human evaluation(Likert Scale), LLM-as-Judge(G-Eval 등)

실무에서 직접 코드로 계산하는 일이 많은 두 가지(Perplexity, BLEU)만 코드로 확인한다. ROUGE/GLUE/MMLU 등은 보통 라이브러리(`evaluate`, `rouge-score`)나 벤치마크 실행 스크립트를 그대로 가져다 쓰므로 개념만 알아두면 충분하다.

In [ ]:
# Perplexity: 모델이 다음 단어를 예측할 때 얼마나 "혼란스러워하는지"를 수치화한 지표. 낮을수록 좋음.
# PPL(W) = exp( -(1/N) * sum(log P(w_i | w_1..i-1)) )
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

def calculate_perplexity(text):
  # 문장을 토큰화
  input_ids = tokenizer.encode(text, return_tensors="pt")

  with torch.no_grad():
    # labels=input_ids를 넣으면 GPT2LMHeadModel이 내부적으로 다음 토큰 예측 loss(cross-entropy)를 계산해줌
    outputs = model(input_ids, labels=input_ids)
    loss = outputs.loss  # 평균 negative log-likelihood

  # Perplexity = exp(평균 negative log-likelihood)
  perplexity = torch.exp(loss)
  return perplexity.item()

natural_sentence = "The weather is nice today and I want to go for a walk."
awkward_sentence = "Purple mathematics quickly sleeps between the refrigerator."

print(f"자연스러운 문장 Perplexity: {calculate_perplexity(natural_sentence):.2f}")
print(f"어색한 문장 Perplexity:     {calculate_perplexity(awkward_sentence):.2f}")
# -> 모델이 이해하기 쉬운(자주 본 패턴의) 문장일수록 Perplexity가 낮게 나옴


In [ ]:
# BLEU: 기계번역 결과가 사람 번역과 얼마나 비슷한지(n-gram 겹침 정도)를 [0, 1]로 측정. 0.4 이상이면 우수.
# 단어 재현 능력만 볼 뿐, 의미적 유사도는 전혀 고려하지 않는다는 한계가 있음.
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
# 참고: 여기서는 이미 단어 단위로 쪼개진 리스트를 직접 넣으므로 nltk의 문장 토크나이저(punkt)는 필요 없음

reference = [["the", "cat", "is", "sitting", "on", "the", "mat"]]  # 정답(사람 번역), 여러 개 줄 수도 있음
candidate_good = ["the", "cat", "is", "sitting", "on", "the", "mat"]      # 모델 번역(정답과 동일)
candidate_bad = ["a", "dog", "was", "lying", "under", "a", "table"]       # 모델 번역(전혀 다름)

# SmoothingFunction: 짧은 문장에서 n-gram이 하나도 안 겹칠 때 점수가 0으로 튀는 걸 완화
smoothie = SmoothingFunction().method1

score_good = sentence_bleu(reference, candidate_good, smoothing_function=smoothie)
score_bad = sentence_bleu(reference, candidate_bad, smoothing_function=smoothie)

print(f"정답과 동일한 번역의 BLEU 점수: {score_good:.4f}  (1.0에 가까움)")
print(f"전혀 다른 번역의 BLEU 점수:     {score_bad:.4f}  (0에 가까움)")


## 14. 압축 요약 (여기까지 왔으면 이것만 기억하자)

- **Self-Attention**: 문장 속 모든 토큰이 서로에게 "얼마나 중요한지" 점수를 매기고(Q·K 내적, $\sqrt{d_k}$로 스케일링, softmax) 그 가중치로 Value를 합쳐서 문맥 반영 벡터를 만든다. RNN처럼 순차 처리가 아니라 한 번에 계산 가능해서 병렬화가 되고, 이게 학습 속도의 핵심 이유다.
- **Multi-Head Attention**: 같은 계산을 여러 관점(head)으로 나눠서 동시에 수행 후 합침 = 앙상블 효과.
- **Positional Encoding**: 순서 정보가 없는 구조적 한계를 sin/cos 고정 벡터를 더해서 보완.
- **Transformer = Encoder(자기 문장 이해) + Decoder(Masked Self-Attention으로 타깃 문장 생성 + Cross-Attention으로 소스 문장 참고)**.
- **BERT(Encoder-only)**: 양방향 문맥 이해가 필요한 분류/유사도/임베딩 작업에 사용. MLM+NSP로 사전학습.
- **GPT(Decoder-only)**: Autoregressive(자기회귀)하게 다음 토큰을 순서대로 생성. 현존 LLM 대부분이 이 구조.
- **T5(Encoder-Decoder)**: 모든 NLP 문제를 "텍스트 넣으면 텍스트 나옴" 형태로 통일, Prefix로 태스크를 구분.
- **LLM 학습**: Pretraining(비지도) → SFT(지도, 사람이 만든 Q&A) → Alignment/RLHF(사람 선호도 반영).
- **평가**: 통계 기반(Perplexity, BLEU, ROUGE) / 벤치마크(SQuAD, MMLU, GLUE) / 직접 평가(Human, LLM-as-Judge).